In [ ]:

import os
from Functions import *
os.chdir('/project/galaxies') #TJ change working directory to be the parent directory
_, filter_files = collect_M51_image_and_filter_files(filter_directory, image_directory)
from astropy.table import Table
import glob
from matplotlib.path import Path
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np
#TJ To create table of galaxies, run tjuchau/projects/EW_vs_Age/Create_cluster_table.py
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
def get_EW_using_filters(feature_filter_file, continuum_filter_files, location, radius):
    #TJ load files

    #TJ get all 3 image fluxes
    Fnu_feature = get_image_flux(feature_filter_file, location, radius, replace_negatives=False)
    Fnu_cont = [get_image_flux(f, location, radius, replace_negatives=False) for f in continuum_filter_files]
    feature_filter = extract_filter_name(feature_filter_file)
    continuum_filters = [extract_filter_name(x) for x in continuum_filter_files]
    if Fnu_feature.unit != Fnu_cont[0].unit:
        print('units are not the same in the feature image and continuum image!')
    elif Fnu_feature.unit == u.W/(u.m**2*u.Hz):
            
        #TJ look up pivot wavelengths
        pivot_feat = jwst_pivots[feature_filter]
        pivot_cont = [jwst_pivots[extract_filter_name(f)] for f in continuum_filter_files]
        
        #TJ convert continuum levels into F_lambda using pivot wavelengths, still need to multiply by dlamda
        fλ_cont = [(Fnu * c / pivot**2).to(u.W / u.m**2 / u.m)
                   for Fnu, pivot in zip(Fnu_cont, pivot_cont)]
        
        #TJ get mean wavelengths
        cont_wls = [jwst_means[f] for f in continuum_filters]
        line_wl = jwst_means[feature_filter]
    
        #TJ interpolate continuum values to the feature wavelength
        feature_continuum = np.interp(
            line_wl.value,
            [w.value for w in cont_wls],
            [f.value for f in fλ_cont]
        ) * u.W / u.m**2 / u.m
        
        #TJ print continuum if needed
        #print("F_lamda of photo continuum : ", feature_continuum)
        #TJ get filter transmission curve info
        wl, T = get_filter_data(feature_filter)
    
        #TJ multiply feature F_lambda by dlambda to complete unit conversion
        norm = np.trapezoid(T, wl) / np.max(T)
        cont_in_filter = feature_continuum * norm
        
        #TJ convert feature filter's F_nu into F_lamda
        fλ_feature = ((Fnu_feature * c / pivot_feat**2).to(u.W / u.m**2 / u.m))*norm 
    
        #TJ feature area is only the area above the continuum
        feature_only = fλ_feature - cont_in_filter
    
        #TJEquivalent width is this area divided by the continuum level
        EW = (feature_only / feature_continuum).to(u.m)
    
        return EW
table = table[np.isfinite(table['EW_187'])]
regions = glob.glob('/project/galaxies/tjuchau/data_files/misc_data/*.deg.reg')
m51_clusters = Table.read('/project/galaxies/tjuchau/data_files/misc_data/clusters.csv')


def read_ds9_region(filename):
    shapes = []

    with open(filename) as f:
        for line in f:
            line = line.strip()

            if line.startswith("polygon"):
                coords = line[8:-1]
                vals = np.array(coords.split(","), dtype=float)
                ra = vals[0::2]
                dec = vals[1::2]
                shapes.append(("polygon", np.column_stack((ra, dec))))

            if line.startswith("box"):
                vals = line[4:-1].split(",")

                ra = float(vals[0])
                dec = float(vals[1])

                width = float(vals[2].replace('"','')) / 3600
                height = float(vals[3].replace('"','')) / 3600
                angle = float(vals[4])

                shapes.append(("box", (ra, dec, width, height, angle)))

    return shapes


def points_in_box(points, ra_c, dec_c, width, height, angle):
    """
    points: Nx2 array of RA,Dec
    width/height in degrees
    angle in degrees
    """

    # offsets
    dx = (points[:,0] - ra_c) * np.cos(np.radians(dec_c))
    dy = (points[:,1] - dec_c)

    theta = np.radians(angle)

    xr = dx*np.cos(theta) + dy*np.sin(theta)
    yr = -dx*np.sin(theta) + dy*np.cos(theta)

    return (np.abs(xr) <= width/2) & (np.abs(yr) <= height/2)


# Load regions
all_shapes = []
for f in regions:
    all_shapes.extend(read_ds9_region(f))


points = np.column_stack((m51_clusters["ra_gaia"], m51_clusters["dec_gaia"]))
inside_any = np.zeros(len(points), dtype=bool)


for shape in all_shapes:

    if shape[0] == "polygon":
        path = Path(shape[1])
        inside_any |= path.contains_points(points)

    if shape[0] == "box":
        ra, dec, w, h, ang = shape[1]
        inside_any |= points_in_box(points, ra, dec, w, h, ang)


region_id = np.full(len(points), -1)

for i, shape in enumerate(all_shapes):

    if shape[0] == "polygon":
        mask = Path(shape[1]).contains_points(points)

    if shape[0] == "box":
        ra, dec, w, h, ang = shape[1]
        mask = points_in_box(points, ra, dec, w, h, ang)

    region_id[mask] = i

m51_clusters["region_id"] = region_id
clusters_in_regions = m51_clusters[inside_any]
clusters_in_regions

In [ ]:
keep_cols = ['col0', 'galaxy', 'best.attenuation.A550', 'best.nebular.logU', 
'best.sfh.age', 'best.stellar.age_m_star', 'best.universe.luminosity_distance',
'best.universe.redshift', 'best.stellar.m_gas', 'best.stellar.m_star', 'ra',
'dec', 'EW_658', 'EW_187', 'radius']
tab = table[keep_cols]
new_row = [155, 'ngc5182', None, None, 3, 150, 5.8e23, 0, 7000, 5000, 55.3, -47.2, 1.7e-8, 1.5e-10, 0.16]
tab.add_row(new_row)
tab

In [ ]:
radius = max(table['radius'])*u.arcsec
for i, row in enumerate(table[15:20]):
    gal_name = row['galaxy']
    loc = [row['ra'], row['dec']]
    image_files = glob.glob(f'tjuchau/data_files/JWST/images/{gal_name.upper()}/*')
    print(i)
    try:
        show_images(image_files, loc, radius, cmap = 'rainbow')
        for image_file in image_files:
            print(get_image_flux(image_file, loc, radius))
    except:
        print('Image does not cover this location')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, unique

# Columns
gal = table['galaxy']
age = table['best.sfh.age']
ew  = table['EW_187']/(table['best.stellar.m_gas']+table['best.stellar.m_star'])

# Container lists
gal_list = []
age_list = []
ew_mean = []
ew_std = []

# Loop through galaxies
for g in np.unique(gal):

    gal_mask = gal == g
    ages_in_gal = np.unique(age[gal_mask])

    for a in ages_in_gal:

        mask = (gal == g) & (age == a)

        ew_vals = ew[mask]

        if len(ew_vals) > 0:
            gal_list.append(g)
            age_list.append(a)
            ew_mean.append(np.mean(ew_vals))
            ew_std.append(np.std(ew_vals))

gal_list = np.array(gal_list)
age_list = np.array(age_list)
ew_mean = np.array(ew_mean)
ew_std = np.array(ew_std)

# Plot
plt.figure()

for g in np.unique(gal_list):

    mask = gal_list == g

    plt.errorbar(
        age_list[mask],
        ew_mean[mask],
        yerr=ew_std[mask],
        fmt='o',
        label=g,
        capsize=3
    )

plt.xlabel("Cluster Age")
plt.ylabel("Paα Equivalent Width")

plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

x = table['best.sfh.age']
y = table['EW_658']/table['EW_187']
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(x.name)
plt.ylabel('NII/Pa-a')
plt.show()

x = table['best.sfh.age']
y = table['EW_658']
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(x.name)
plt.ylabel('NII')
plt.show()

x = table['best.sfh.age']
y = (table['EW_187']/table['EW_658'])/(table['best.stellar.m_gas'])
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.xlabel(x.name)
plt.ylabel('Pa-a/mass')
plt.show()

x = table['best.sfh.age']
y = (table['EW_187'])
plt.scatter(x, y, s = 5, color = 'blue')
plt.xscale('log')
plt.xlabel(x.name)
plt.ylabel('Pa-a')
plt.show()

In [ ]:
import pickle
galaxy_tables['ngc1433'][7]['radius'] = 0.4
with open("/project/galaxies/tjuchau/data_files/misc_data/saved_data/Kiana_galaxies.pkl", "wb") as file:
    pickle.dump(galaxy_tables, file)

In [ ]:
young_clusters = kiana_table[kiana_table['EW_658'] > 8e-9]
image_files = glob.glob(f'tjuchau/data_files/JWST/images/NGC1433/*')
ages = []
ews = []
for row in young_clusters:
    ra = row['ra']
    dec = row['dec']
    age = row['best.sfh.age']
    ages.append(age)
    pa_EW = get_EW_using_filters(image_files[1], [image_files[0], image_files[2]], [ra,dec], 0.5*u.arcsec).value
    ews.append(pa_EW)
    print(f'Age : {age}, Pa-EW : {pa_EW}')
    try:
        show_images(image_files, [ra,dec], 0.5*u.arcsec, cmap = 'rainbow')
    except:
        print('Image does not cover this location')
plt.scatter(ages, ews)
plt.show()

In [ ]:
name = 'ngc1433'
ngc1433_bad = [0,4,6,9,10]
ngc1433_maybe = [7]
ngc1433_good = np.setdiff1d(np.arange(len(galaxy_tables[name])), np.concatenate([ngc1433_bad, ngc1433_maybe]))
image_files = glob.glob(f'tjuchau/data_files/JWST/images/{name.upper()}/*')
for i, row in enumerate(galaxy_tables[name][:100]):
    if i in ngc1433_good:
        location = row['location']
        print(i)
        show_images(image_files, location, row['radius']*u.arcsec, cmap = 'rainbow')

In [ ]:
for i, row in enumerate(galaxy_tables[name][ngc1433_bad]):
    location = row['location']
    print(ngc1433_bad[i])
    show_images(image_files, location, 0.3*u.arcsec, cmap = 'rainbow')

In [ ]:
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
table